# 박스 시퀀스 vs 팔레트 용량 분석

**목적** — `templete code/box_sequence` 의 입력 박스들을 우리 팔레트(bin)에 **애초에 다 담을 수 없다**는 점을 정량적으로 확인한다.

핵심 질문:
1. 박스 전체 부피의 합이 팔레트 부피를 얼마나 초과하는가?
2. 부피가 100%로 완벽히 채워진다고 가정해도(이론적 상한) 몇 번째 박스에서 용량을 넘어서는가?
3. 현실적인 패킹 효율을 감안하면 실제로 몇 개나 적재 가능한가?
4. 박스 크기·질량 분포는 어떤가?

> 분석 전용 노트북이다. 제출 코드와 무관하므로 pandas/matplotlib 등을 자유롭게 사용한다.

In [1]:
import json
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정 ───────────────────────────────────────────────
# 팔레트(bin) 크기 (m) — config/algorithm_config.yaml 및 과제 규격 기준
#   가로(length, X) 1.2 / 세로(width, Y) 1.0 / 최대높이(height, Z) 1.25
PALLET = {"length": 1.2, "width": 1.0, "height": 1.25}
PALLET_VOLUME = PALLET["length"] * PALLET["width"] * PALLET["height"]  # 1.5 m^3

# box_sequence 폴더 경로 (필요시 수정)
BOX_DIR_CANDIDATES = [
    "templete code/box_sequence",
    "box_sequence",
    "./box_sequence",
]
BOX_DIR = next((p for p in BOX_DIR_CANDIDATES if os.path.isdir(p)), BOX_DIR_CANDIDATES[0])

print(f"팔레트 크기      : {PALLET['length']} x {PALLET['width']} x {PALLET['height']} m")
print(f"팔레트 부피      : {PALLET_VOLUME:.4f} m^3")
print(f"box_sequence 경로: {BOX_DIR}")

ModuleNotFoundError: No module named 'pandas'

## 1. 데이터 로드

`box_sequence` 파일은 한 줄에 박스 하나씩인 **JSONL** 형식이다(드물게 JSON 배열일 수 있어 둘 다 처리). 각 박스는 `step, id, size=[L, W, H], mass` 등을 가진다.

In [ ]:
def load_boxes(path):
    """JSONL 또는 JSON 배열 파일을 박스 리스트로 로드."""
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    if not text:
        return []
    if text[0] == "[":
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]


def box_volume(b):
    L, W, H = b["size"]
    return float(L) * float(W) * float(H)


files = sorted(glob.glob(os.path.join(BOX_DIR, "*.json")))
datasets = {os.path.basename(f): load_boxes(f) for f in files}

for name, boxes in datasets.items():
    print(f"{name:24s}  박스 수 = {len(boxes)}")

# 첫 파일 미리보기
first = next(iter(datasets))
pd.DataFrame(datasets[first]).head()

## 2. 용량 대비 전체 부피

각 데이터셋의 박스 부피 총합을 팔레트 부피(1.5 m³)와 비교한다. 비율이 1을 넘으면 **물리적으로 전부 담는 것이 불가능**하다는 뜻이다.

In [ ]:
rows = []
for name, boxes in datasets.items():
    vols = np.array([box_volume(b) for b in boxes])
    total = vols.sum()
    rows.append({
        "dataset": name,
        "박스수": len(boxes),
        "총부피(m^3)": round(total, 4),
        "팔레트부피(m^3)": round(PALLET_VOLUME, 4),
        "초과배율": round(total / PALLET_VOLUME, 3),
        "평균박스부피(m^3)": round(vols.mean(), 5),
    })

summary = pd.DataFrame(rows)
display(summary)

print()
for r in rows:
    print(f"[{r['dataset']}] 총 부피가 팔레트의 {r['초과배율']}배 "
          f"→ 부피만 봐도 전부 적재 불가능 (이상적으로 채워도 약 {round(1/r['초과배율']*100,1)}%만 수용)")

In [ ]:
# 막대그래프: 데이터셋별 총 박스 부피 vs 팔레트 부피
fig, ax = plt.subplots(figsize=(7, 4))
names = [r["dataset"] for r in rows]
totals = [r["총부피(m^3)"] for r in rows]
ax.bar(names, totals, color="#4C72B0", label="박스 총 부피")
ax.axhline(PALLET_VOLUME, color="#C44E52", linestyle="--", linewidth=2,
           label=f"팔레트 부피 = {PALLET_VOLUME:.2f} m³")
ax.set_ylabel("부피 (m³)")
ax.set_title("데이터셋별 박스 총 부피 vs 팔레트 용량")
ax.legend()
for i, v in enumerate(totals):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.show()

## 3. 누적 부피 곡선 — 도착 순서 기준 "이론적 상한"

박스를 도착 순서(step)대로 누적했을 때, 누적 부피가 팔레트 부피를 넘어서는 지점을 찾는다.  
**틈 없이 100%로 완벽히 채운다는(현실에선 불가능한) 가정**에서의 상한이므로, 실제로 적재 가능한 개수는 이보다 **반드시 더 적다.**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for name, boxes in datasets.items():
    vols = np.array([box_volume(b) for b in boxes])
    cum = np.cumsum(vols)
    ax.plot(np.arange(1, len(cum) + 1), cum, label=name)

    # 팔레트 부피를 넘어서는 첫 박스 index
    over = np.argmax(cum > PALLET_VOLUME)
    if cum[over] > PALLET_VOLUME:
        ax.axvline(over + 1, color="gray", linestyle=":", alpha=0.6)
        ax.text(over + 1, PALLET_VOLUME * 0.4, f"  {name}:\n  {over+1}번째 초과",
                fontsize=8, color="gray")

ax.axhline(PALLET_VOLUME, color="#C44E52", linestyle="--", linewidth=2,
           label=f"팔레트 부피 = {PALLET_VOLUME:.2f} m³")
ax.set_xlabel("도착 순서 기준 누적 박스 개수")
ax.set_ylabel("누적 부피 (m³)")
ax.set_title("누적 박스 부피 (100% 완벽 패킹 가정 = 이론적 상한)")
ax.legend()
plt.tight_layout()
plt.show()

for name, boxes in datasets.items():
    vols = np.array([box_volume(b) for b in boxes])
    cum = np.cumsum(vols)
    over = int(np.argmax(cum > PALLET_VOLUME)) + 1
    print(f"[{name}] 도착 순서대로 채울 때 {over}번째 박스에서 부피 한도 초과 "
          f"(즉, 100% 패킹이 가능해도 최대 {over-1}개까지만 — 현실은 그보다 적음)")

## 4. 박스 크기·질량 분포

과제 스펙(W·L 170~320 mm, H 130~260 mm, 질량 0.5~6.0 kg) 범위 안에 데이터가 들어오는지, 어떤 분포인지 본다. 평가셋은 매번 랜덤 생성되므로 **개별 파일이 아니라 이 분포 전체에 강건한 전략**이 필요하다.

In [ ]:
# 모든 데이터셋을 합쳐 분포 확인
all_boxes = [b for boxes in datasets.values() for b in boxes]
df = pd.DataFrame(all_boxes)
df[["L", "W", "H"]] = pd.DataFrame(df["size"].tolist(), index=df.index)
df["vol"] = df["L"] * df["W"] * df["H"]

print("전체 박스 수:", len(df))
display(df[["L", "W", "H", "mass", "vol"]].describe().round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
specs = [
    ("L", "길이 L (m)", (0.17, 0.32)),
    ("W", "폭 W (m)", (0.17, 0.32)),
    ("H", "높이 H (m)", (0.13, 0.26)),
    ("mass", "질량 (kg)", (0.5, 6.0)),
    ("vol", "박스 부피 (m³)", None),
]
for ax, (col, label, spec) in zip(axes.flat, specs):
    ax.hist(df[col], bins=25, color="#55A868", edgecolor="white")
    ax.set_title(label)
    if spec is not None:
        ax.axvline(spec[0], color="#C44E52", linestyle=":", label="스펙 범위")
        ax.axvline(spec[1], color="#C44E52", linestyle=":")
        ax.legend(fontsize=8)

# type_name 분포 (있을 경우)
ax = axes.flat[5]
if "type_name" in df.columns:
    df["type_name"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#8172B3")
    ax.set_title("박스 type 분포")
else:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. 현실적 적재 개수 추정 (패킹 효율 반영)

실제 박스 적재는 틈(void) 때문에 부피의 100%를 못 채운다. 직육면체 박스 + 회전 0/90도만 허용되는 환경에서는 통상 **패킹 효율 60~75%** 수준이 현실적이다. 효율 `η`를 가정하면 수용 가능한 박스 부피는 `1.5 × η`, 적재 가능 개수 ≈ `(1.5 × η) / 평균박스부피`.

In [ ]:
mean_vol = df["vol"].mean()

rows = []
for eta in [0.55, 0.60, 0.65, 0.70, 0.75, 1.00]:
    usable = PALLET_VOLUME * eta
    est_n = usable / mean_vol
    rows.append({
        "패킹효율 η": f"{eta:.0%}" + ("  (이론 상한)" if eta == 1.0 else ""),
        "수용부피(m³)": round(usable, 3),
        "적재가능 박스수(추정)": int(est_n),
        "전체250개중 비율": f"{est_n/250:.0%}",
    })

est = pd.DataFrame(rows)
display(est)

print(f"\n평균 박스 부피 = {mean_vol:.5f} m³")
print("→ 250개 중 대부분은 적재 대상이 못 되며, '어떤 박스를 골라 안정적으로 채우느냐'가 본질적 문제.")

## 결론

- 데이터셋의 박스 총 부피는 팔레트 부피(1.5 m³)의 **약 1.8~1.9배**라서, 부피만 따져도 전부 적재하는 것은 불가능하다.
- 틈 없이 100%로 채운다는 비현실적 가정에서도 **절반 안팎**의 박스만 들어가고, 현실적 패킹 효율(60~75%)을 적용하면 **그보다 더 적다.**
- 따라서 과제의 본질은 "모든 박스 적재"가 아니라, **버퍼 윈도우 안에서 어떤 박스를 골라 1.5 m³ 한도 안에 안정적으로(붕괴·이탈·높이초과 없이) 채워 적재율을 극대화할지** 선택·배치하는 최적화 문제다.
- 평가셋은 매번 새로 랜덤 생성되므로, 특정 파일이 아니라 위에서 본 **크기·질량 분포 전반에 강건한 전략**을 설계해야 한다.